# BBBP general local-BFS — Q=T scan

This notebook trains QRC, the aligned old scalar-state CRC, and a flattened-token linear baseline for `Q=T=1,...,24`. Each setting uses ten paired seeds (`0,10,...,90`), a seed-dependent scaffold split, seed-dependent reservoir parameters, 500 epochs, and validation-only checkpoint selection (`AUC > loss > ACC`).

In [ ]:
from pathlib import Path
import sys
import torch

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'core' / 'linear_readout.py').exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError('Could not locate 2_linear_readout project root')
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from core.linear_readout import (
    DEFAULT_SEEDS, ExperimentConfig, prepare_q_equal_t_layout,
    run_scan, collect_scan_results, plot_q_equal_t_scan,
)
print('project:', PROJECT_ROOT)
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())

## Fixed protocol

The BFS token design is task-independent and identical in form to BACE. QRC uses `W=0.4`, `g=0.84`, `L=3`; QRC/CRC consume the same `tanh`-bounded `T x 3Q` tensor and return the same `Q(Q+1)/2` feature dimension.

In [ ]:
SCAN_ROOT = PROJECT_ROOT / 'general' / 'BBBP' / '01_Q_equal_T_scan'
DATA_DIR = PROJECT_ROOT / 'general' / 'BBBP' / 'data'
SETTINGS = prepare_q_equal_t_layout(SCAN_ROOT)
CFG = ExperimentConfig(
    dataset='BBBP', data_dir=str(DATA_DIR), scan_root=str(SCAN_ROOT),
    epochs=500, seeds=DEFAULT_SEEDS, W=0.4, g=0.84, layers=3,
    qrc_devices=('cuda:0', 'cuda:1'), run_qrc=True, force=True,
)
print(CFG)
print('settings:', len(SETTINGS), SETTINGS[0], '...', SETTINGS[-1])

## Train and evaluate

The run is resumable. QRC seeds are divided between `cuda:0` and `cuda:1`; CRC and raw use the same split and linear-readout protocol for each seed.

In [ ]:
results = run_scan(CFG, SETTINGS)
results.tail(12)

## Result visualization

Lines show the ten-seed mean; shading is one standard deviation. The figure and source data are written into this scan folder.

In [ ]:
summary = collect_scan_results(SCAN_ROOT)
display(summary)
fig = plot_q_equal_t_scan(SCAN_ROOT, 'BBBP')
fig